# RF-DETR Complete Local Inference Pipeline
This notebook demonstrates a modular, production-style setup for running and evaluating RF-DETR models locally.


## 1. Environment Verification
Check CUDA, PyTorch, and required libraries.


In [1]:
import sys
import os
sys.path.append('..') # Ensure we can import from utils

import torch
import cv2
import supervision as sv

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"Supervision Version: {sv.__version__}")


PyTorch Version: 2.11.0+cu126
CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
OpenCV Version: 4.13.0
Supervision Version: 0.27.0.post2


## 2. Directory Initialization
Automatically create required folders: datasets, models, logs, outputs.


In [2]:
directories = [
    "../datasets/input_images",
    "../datasets/input_videos",
    "../datasets/test_images",
    "../datasets/outputs/images",
    "../datasets/outputs/videos",
    "../datasets/outputs/metrics",
    "../models/pretrained",
    "../logs"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Verified directory: {directory}")


Verified directory: ../datasets/input_images
Verified directory: ../datasets/input_videos
Verified directory: ../datasets/test_images
Verified directory: ../datasets/outputs/images
Verified directory: ../datasets/outputs/videos
Verified directory: ../datasets/outputs/metrics
Verified directory: ../models/pretrained
Verified directory: ../logs


## 3. Load Pretrained RF-DETR Model
Initialize the model using our wrapper. This handles offline caching and device mapping.


In [3]:
from utils.inference import RFDETRInference

# Initialize the Medium model. Change size="N", "S", "L", "XL" as needed.
model_wrapper = RFDETRInference(size="M", config_path="../configs/config.json")
print("Model loaded successfully.")


Loading RF-DETR Size: M on cuda...
[2026-05-09 17:37:51] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


[2026-05-09 17:37:51] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-09 17:37:51] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-09 17:37:52] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.
Model loaded successfully.


## 4. Visualization & 5. FPS Metrics Setup
Import visualization and performance tracking utilities.


In [4]:
from utils.visualization import Visualizer
from utils.fps import PerformanceTracker

visualizer = Visualizer()
tracker = PerformanceTracker()


## 6. Image Inference Pipeline
Process a single image, track inference time, and save the result.


In [5]:
from PIL import Image
import numpy as np

# Create a dummy image for validation if none exists
dummy_image_path = "../datasets/input_images/images.jpg"
if not os.path.exists(dummy_image_path):
    img = np.zeros((480, 640, 3), dtype=np.uint8)
    cv2.putText(img, "Sample Image", (200, 240), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    cv2.imwrite(dummy_image_path, img)

image = cv2.imread(dummy_image_path)

tracker.start()
detections = model_wrapper.predict(image, threshold=0.5)
tracker.stop()

# Visualize
annotated_image = visualizer.annotate(image, detections)

# Save output
output_path = "../datasets/outputs/images/output_image.jpg"
cv2.imwrite(output_path, annotated_image)

print(f"Saved annotated image to {output_path}")
tracker.log_metrics()


[2026-05-09 17:37:53] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[2026-05-09 17:37:54] [WARNING] rf-detr - predict() encountered class_id values out of range [0, 80]: [np.int64(85)] — mapping to empty string


Saved annotated image to ../datasets/outputs/images/output_image.jpg
FPS: 1.18 | Latency: 844.07ms | GPU Mem (Alloc): 138.3MB


## 7. Video Inference Pipeline
Run frame-by-frame inference on a video and generate an annotated output video.


In [6]:
from utils.video_utils import VideoProcessor

video_input = "../datasets/input_videos/video.mp4"
video_output = "../datasets/outputs/videos/output_video.mp4"

# Create a dummy video for validation if none exists
if not os.path.exists(video_input):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_input, fourcc, 30.0, (640, 480))
    for i in range(30):
        img = np.zeros((480, 640, 3), dtype=np.uint8)
        cv2.circle(img, (100 + i*10, 240), 50, (0, 0, 255), -1)
        out.write(img)
    out.release()

processor = VideoProcessor(video_input)

# Define callbacks
def process_frame(frame):
    return model_wrapper.predict(frame, threshold=0.5)

def visualize_frame(frame, detections):
    return visualizer.annotate(frame, detections)

# Note: Set output_path to None if you just want to run without saving
processor.process_video(
    inference_callback=process_frame,
    visualization_callback=visualize_frame,
    output_path=video_output
)


Starting video processing from source: ../datasets/input_videos/video.mp4
Video processing completed.


## 8. Webcam / Live Camera Inference
Read from webcam, perform real-time detection, and display metrics.


In [17]:
# Note: Set webcam_id to 0 for default camera.
# This cell is commented out to prevent CI/CD or headless environment hangs.

webcam_id = 0
cap = cv2.VideoCapture(webcam_id)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    tracker.start()
    detections = model_wrapper.predict(frame, threshold=0.5)
    tracker.stop()
    
    annotated_frame = visualizer.annotate(frame, detections)
    
    # Render FPS
    metrics = tracker.get_metrics()
    cv2.putText(annotated_frame, f"FPS: {metrics['fps']:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("RF-DETR Live", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cap.release()
cv2.destroyAllWindows()

# print("Webcam inference code is ready. Uncomment to run locally.")


## 9. Evaluation Metrics
Initialize the COCO evaluation framework for validation on test datasets.


In [ ]:
# from utils.metrics import COCOEvaluator

# ground_truth_json = "../datasets/test_images/annotations.json"
# evaluator = COCOEvaluator(ground_truth_json)

# # Note: Once you have a populated dataset, you can loop through images,
# # gather `model_predictions` dict mapping image_name -> sv.Detections
# # and run `evaluator.evaluate(model_predictions)`.
# print("Evaluation framework initialized.")


Failed to load COCO dataset from ../datasets/test_images/annotations.json: [Errno 2] No such file or directory: '../datasets/test_images/annotations.json'
Evaluation framework initialized.


## 10. Logging System
Save metrics and inference summaries to the logs directory.


In [9]:
import json
import datetime

log_data = {
    "timestamp": str(datetime.datetime.now()),
    "model_size": model_wrapper.size,
    "performance": tracker.get_metrics()
}

log_path = "../logs/execution_summary.json"
with open(log_path, "w") as f:
    json.dump(log_data, f, indent=4)
    
print(f"Logged execution summary to {log_path}")


Logged execution summary to ../logs/execution_summary.json


## 11. Error Handling
Demonstrate robust handling of missing files and invalid inputs.


In [10]:
try:
    print("Testing error handling on missing image...")
    # Passing None to simulate failed imread
    detections = model_wrapper.predict(None)
    if detections is None:
        print("Model handled invalid input safely (returned None).")
except Exception as e:
    print(f"Caught exception: {e}")


Testing error handling on missing image...
Error during prediction: unsupported operand type(s) for *: 'NoneType' and 'int'
Model handled invalid input safely (returned None).
